# Homework 1 | Handling Data

A wise figure once said to me: to do anything useful in life, you need to be able to handle 8 gigabytes of option data. The dataset you all have is SPX option data from 1996 to 2023. FUN FACT: I recieved this data as a part of a Summer Reserach Experience I did; the project was paid for, but I found it quite interesting that this SPX daily data costed 3,000 dollars...

In order to really get crack-a-lackin with this dataset, we need a systematic and efficient way to access the contents. Right now the data is stored in a bunch of .zip files on your computers. Your job will be to get the data from a .zip format into a .pkl format (pickle)

Why do this? Well, .zip formats need to be unzipped in order to be accessed. So that's why.

As great thinkers, we will NOT be unzipping the 3,000+ contents of the folder I gave you one by one... Instead, you will make Python do this for you!

## Tasks (not necessarily in order)

### 1. Unzip each of the files and place the result in a seperate folder

### 2. For each unzipped file, convert it to a pkl format

### 3. Rename each file --> SPX_OPTIONS_MMDDYYYY

### 4. Open up a file and play around with the data :)


## Your work starts here

In [19]:
import os
import re
import zipfile
from datetime import datetime
import pandas as pd
import pickle

In [22]:
# PATHS
zip_folder = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip"
unzipped_folder = os.path.join(zip_folder, "unzipped")
pickle_folder = os.path.join(zip_folder, "pickles")
os.makedirs(unzipped_folder, exist_ok=True)
os.makedirs(pickle_folder, exist_ok=True)

In [23]:
def unzip_all_files():
    """Unzip all .zip files into separate folders (safe to re-run)."""
    for file in os.listdir(zip_folder):
        if file.lower().endswith(".zip"):
            src = os.path.join(zip_folder, file)
            dest = os.path.join(unzipped_folder, os.path.splitext(file)[0])
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(src, "r") as zf:
                zf.extractall(dest)
            print(f"Unzipped {file} into {dest}")

In [15]:
def convert_to_pickle():
    """Convert each CSV in the unzipped folders to a pickle file with proper naming."""
    for root, _, files in os.walk(unzipped_folder):
        for file in files:
            if file.endswith(".csv"):
                csv_path = os.path.join(root, file)
                try:
                    df = pd.read_csv(csv_path)

                    # Extract date from filename if possible
                    # Example assumes filename has YYYYMMDD in it somewhere
                    date_str = ''.join(filter(str.isdigit, file))
                    try:
                        dt = datetime.strptime(date_str, "%Y%m%d")
                        formatted_name = f"SPX_OPTIONS_{dt.strftime('%m%d%Y')}.pkl"
                    except ValueError:
                        # fallback if no date found
                        formatted_name = f"SPX_OPTIONS_{file.replace('.csv', '')}.pkl"

                    pkl_path = os.path.join(pickle_folder, formatted_name)

                    # Save as pickle
                    with open(pkl_path, "wb") as f:
                        pickle.dump(df, f)

                    print(f"Converted {file} → {formatted_name}")

                except Exception as e:
                    print(f"Failed to convert {file}: {e}")

In [24]:
def parse_date_from_name(name: str) -> str:
    """
    Return 'MMDDYYYY' from the filename.
    - Prefer 8-digit YYYYMMDD if present.
    - Else accept 6-digit YYYYMM and assume day=01.
    - Else return the raw stem (used only as a last resort).
    """
    # 8-digit date
    m8 = re.search(r"(?<!\d)(20|19)\d{6}(?!\d)", name)
    if m8:
        dt = datetime.strptime(m8.group(), "%Y%m%d")
        return dt.strftime("%m%d%Y")
    # 6-digit yyyymm
    m6 = re.search(r"(?<!\d)(20|19)\d{4}(?!\d)", name)
    if m6:
        dt = datetime.strptime(m6.group() + "01", "%Y%m%d")
        return dt.strftime("%m%d%Y")
    # fallback (rare)
    stem = os.path.splitext(os.path.basename(name))[0]
    return stem

In [25]:
def read_table_safely(path: str) -> pd.DataFrame:
    """
    Try to read unknown .txt/.csv:
      1) Automatic delimiter inference (sep=None, engine='python')
      2) Try common delimiters
      3) Try whitespace
      4) Fallback to fixed-width (read_fwf)
    """
    # 1) sep=None inference
    try:
        df = pd.read_csv(path, sep=None, engine="python")
        # If it worked and produced multiple cols, great
        if df.shape[1] > 1:
            return df
    except Exception:
        pass

    # 2) common delimiters
    for sep in [",", "\t", "|", ";"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] > 1:
                return df
        except Exception:
            continue

    # 3) whitespace-delimited
    try:
        df = pd.read_csv(path, delim_whitespace=True)
        if df.shape[1] > 1:
            return df
    except Exception:
        pass

    # 4) fixed-width fallback
    try:
        df = pd.read_fwf(path)
        return df
    except Exception as e:
        raise RuntimeError(f"Failed to parse {path}: {e}")

In [26]:
def convert_to_pickle():
    """
    Find .txt/.csv in unzipped/, parse them, and save as pickles:
    SPX_OPTIONS_MMDDYYYY.pkl
    """
    converted = 0
    for root, _, files in os.walk(unzipped_folder):
        for file in files:
            if not file.lower().endswith((".txt", ".csv")):
                continue
            src = os.path.join(root, file)
            date_token = parse_date_from_name(file)
            target_name = f"SPX_OPTIONS_{date_token}.pkl"
            dst = os.path.join(pickle_folder, target_name)

            if os.path.exists(dst):
                print(f"[skip] {target_name} already exists")
                continue

            try:
                df = read_table_safely(src)
                # Optional: try to parse date-like columns
                for c in df.columns:
                    if re.search(r"date|time|dt|timestamp", str(c), re.I):
                        try:
                            df[c] = pd.to_datetime(df[c], errors="ignore")
                        except Exception:
                            pass

                with open(dst, "wb") as f:
                    pickle.dump(df, f, protocol=pickle.HIGHEST_PROTOCOL)

                converted += 1
                print(f"Converted {file}  ->  {target_name} (rows={len(df)}, cols={df.shape[1]})")
            except Exception as e:
                print(f"[error] Could not convert {src}: {e}")
    if converted == 0:
        print("No tables converted — check file formats or permissions.")


In [27]:
def open_sample_pickle():
    """Open one pickle and print a quick summary."""
    pkl_files = sorted([f for f in os.listdir(pickle_folder) if f.lower().endswith(".pkl")])
    if not pkl_files:
        print("No pickle files found.")
        return
    sample = os.path.join(pickle_folder, pkl_files[0])
    with open(sample, "rb") as f:
        df = pickle.load(f)
    print(f"\nOpened sample: {os.path.basename(sample)}")
    print(f"Shape: {df.shape}")
    print("\nColumns & dtypes:")
    print(df.dtypes)
    print("\nHead:")
    print(df.head(10))

In [28]:
if __name__ == "__main__":
    unzip_all_files()       # safe to re-run
    convert_to_pickle()     # will target .txt/.csv
    open_sample_pickle()    # peek at one

Unzipped GI.NA.IVYOPPRCD_200203.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_200203
Unzipped GI.NA.IVYOPPRCD_199705.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_199705
Unzipped GI.NA.IVYOPPRCD_200007.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_200007
Unzipped GI.NA.IVYOPPRCD_202112.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_202112
Unzipped GI.NA.IVYOPPRCD_200302.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_200302
Unzipped GI.NA.IVYOPPRCD_200112.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_200112
Unzipped GI.NA.IVYOPPRCD_199902.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_199902
Converted GI.NA.IVYO

/var/folders/8m/rcncq3bj137bqg7bk01pr0xw0000gn/T/ipykernel_83326/4258706214.py:26: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_datetime(df[c], errors="ignore")
/var/folders/8m/rcncq3bj137bqg7bk01pr0xw0000gn/T/ipykernel_83326/4258706214.py:26: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_datetime(df[c], errors="ignore")
/var/folders/8m/rcncq3bj137bqg7bk01pr0xw0000gn/T/ipykernel_83326/4258706214.py:26: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_datetime(df[c], errors="ignore")
/var/folders/8m/rcncq3bj137bqg7bk01pr0xw0000gn/T/ipykernel_83326/4258706214.py:26: FutureWarning: errors='ignore' is d

Converted GI.NA.IVYOPPRCD_199705.txt  ->  SPX_OPTIONS_05011997.pkl (rows=9314, cols=28)
Converted GI.NA.IVYOPPRCD_199902.txt  ->  SPX_OPTIONS_02011999.pkl (rows=9135, cols=28)


/var/folders/8m/rcncq3bj137bqg7bk01pr0xw0000gn/T/ipykernel_83326/4258706214.py:26: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_datetime(df[c], errors="ignore")
/var/folders/8m/rcncq3bj137bqg7bk01pr0xw0000gn/T/ipykernel_83326/4258706214.py:26: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_datetime(df[c], errors="ignore")


Converted GI.NA.IVYOPPRCD_202112.txt  ->  SPX_OPTIONS_12012021.pkl (rows=502027, cols=28)
Converted GI.NA.IVYOPPRCD_200112.txt  ->  SPX_OPTIONS_12012001.pkl (rows=10174, cols=28)
Converted GI.NA.IVYOPPRCD_200302.txt  ->  SPX_OPTIONS_02012003.pkl (rows=9976, cols=28)

Opened sample: SPX_OPTIONS_02011999.pkl
Shape: (9135, 28)

Columns & dtypes:
SecurityID                    int64
Date                 datetime64[ns]
OptionID                      int64
Exchange                      int64
Currency                      int64
Expiration                    int64
Strike                        int64
CallPut                      object
Symbol                       object
Bid                         float64
Ask                         float64
Last                        float64
Volume                        int64
OpenInterest                  int64
SpecialSettlement             int64
ImpliedVolatility           float64
Delta                       float64
Gamma                       float64
Vega   

/var/folders/8m/rcncq3bj137bqg7bk01pr0xw0000gn/T/ipykernel_83326/4258706214.py:26: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_datetime(df[c], errors="ignore")


In [18]:
def debug_unzipped_files():
    for root, _, files in os.walk(unzipped_folder):
        for file in files:
            print("Found:", os.path.join(root, file))

if __name__ == "__main__":
    unzip_all_files()
    debug_unzipped_files()   # <-- Add this line to see what's inside
    convert_to_pickle()
    open_sample_pickle()


Unzipped GI.NA.IVYOPPRCD_200203.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_200203
Unzipped GI.NA.IVYOPPRCD_199705.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_199705
Unzipped GI.NA.IVYOPPRCD_200007.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_200007
Unzipped GI.NA.IVYOPPRCD_202112.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_202112
Unzipped GI.NA.IVYOPPRCD_200302.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_200302
Unzipped GI.NA.IVYOPPRCD_200112.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_200112
Unzipped GI.NA.IVYOPPRCD_199902.zip into /Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/Options_Data_Zip/unzipped/GI.NA.IVYOPPRCD_199902
Found: /Users/zakdja